<center> <h2><b>3 Representation Learning</b></h2> </center>

In [82]:
import importlib
import sys
sys.path.append("..")  # Ensure the parent directory is in the path

import visualization.data_exploration_plots
import data_preprocessing.data_preprocessing
import models.classic_ml.classic_ml_models

# Reload the module to reflect changes
importlib.reload(visualization.data_exploration_plots)
importlib.reload(data_preprocessing.data_preprocessing)
importlib.reload(models.classic_ml.classic_ml_models)

from data_preprocessing.data_cleaning import *
from models.classic_ml.classic_ml_models import *
from models.lstm_autoencoder.train_evaluate import *
from utils.data_loader import load_and_clean_data

import torch


from models.lstm_rnn.lstm_rnn import preprocess_for_lstm, build_lstm_model
import tensorflow as tf

from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
from keras.utils import set_random_seed

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
set_random_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.mps.is_available() else
                      "cpu")
print(f"Using : {device}")

Using : mps


# Prepare Data

In [83]:
x_paths = ['set-a.parquet', 'set-b.parquet', 'set-c.parquet']
y_paths = ['outcomes-a.parquet', 'outcomes-b.parquet', 'outcomes-c.parquet']

X_train, X_valid, X_test, y_train, y_valid, y_test = load_and_clean_data(x_paths, y_paths, clean_data)

In [85]:
X_train.head(3)

,recordid,time,ALP,ALT,AST,Age,Albumin,BUN,Bilirubin,Cholesterol,...,RespRate,SaO2,SysABP,Temp,TroponinI,TroponinT,Urine,WBC,Weight,pH
0,132539,00:00,NaN,NaN,NaN,54,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,132539,01:00,NaN,NaN,NaN,54,NaN,NaN,NaN,NaN,...,19.0,NaN,NaN,35.6,NaN,NaN,60.0,NaN,NaN,NaN
2,132539,02:00,NaN,NaN,NaN,54,NaN,NaN,NaN,NaN,...,18.0,NaN,NaN,NaN,NaN,NaN,30.0,NaN,NaN,NaN


## Q3.1 Pretraining and Linear Probes

### Pretrain LSTM-autoencoder

In [63]:
X_train, X_valid, X_test = preprocess_for_lstm(X_train, X_valid, X_test)

In [64]:
X_train.shape

(4000, 49, 40)

In [65]:
# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
X_valid_tensor = torch.tensor(X_valid, dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

In [76]:
from models.lstm_autoencoder.lstm_autoencoder import LSTMAE

# Define model parameters
input_size = X_train.shape[2]
hidden_size = 60
dropout_ratio = 0.2
seq_len = X_train.shape[1]
learning_rate = 0.001
epochs = 50
batch_size = 32
patience = 3
clip_value = 1.0

# Create the model instance
model = LSTMAE(input_size=input_size,
               hidden_size=hidden_size,
               dropout_ratio=dropout_ratio,
               seq_len=seq_len)

# Train and evaluate the model
eval_loss_test = train_and_evaluate(
    model,
    X_train_tensor,
    X_valid_tensor,
    X_test_tensor,
    epochs=epochs,
    batch_size=batch_size,
    learning_rate=learning_rate,
    patience=patience,
    clip_val=clip_value
)

# Output the results
print(f"Test Loss: {eval_loss_test:.4f}")

/Users/tnorlha/miniconda3/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/Users/tnorlha/Desktop/ML Healthcare/PhysioNet-ICU-Mortality-Prediction/models/lstm_autoencoder/train_evaluate.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dataset = TensorDataset(torch.tensor(X, dtype=torch.float32))


Epoch 1/50, Train Loss: 0.9697, Valid Loss: 2.3991
Epoch 2/50, Train Loss: 0.9483, Valid Loss: 2.3722
Epoch 3/50, Train Loss: 0.9223, Valid Loss: 2.3539
Epoch 4/50, Train Loss: 0.8976, Valid Loss: 2.3251
Epoch 5/50, Train Loss: 0.8737, Valid Loss: 2.3057
Epoch 6/50, Train Loss: 0.8577, Valid Loss: 2.2924
Epoch 7/50, Train Loss: 0.8421, Valid Loss: 2.2750
Epoch 8/50, Train Loss: 0.8247, Valid Loss: 2.2606
Epoch 9/50, Train Loss: 0.8126, Valid Loss: 2.2497
Epoch 10/50, Train Loss: 0.8025, Valid Loss: 2.2413
Epoch 11/50, Train Loss: 0.7928, Valid Loss: 2.2335
Epoch 12/50, Train Loss: 0.7853, Valid Loss: 2.2266
Epoch 13/50, Train Loss: 0.7783, Valid Loss: 2.2189
Epoch 14/50, Train Loss: 0.7715, Valid Loss: 2.2159
Epoch 15/50, Train Loss: 0.7681, Valid Loss: 2.2106
Epoch 16/50, Train Loss: 0.7631, Valid Loss: 2.2068
Epoch 17/50, Train Loss: 0.7598, Valid Loss: 2.2049
Epoch 18/50, Train Loss: 0.7556, Valid Loss: 2.1995
Epoch 19/50, Train Loss: 0.7517, Valid Loss: 2.1972
Epoch 20/50, Train Lo

### Freeze weights + Compute a single embedding vector for each patient + train a logistic regression

In [71]:
with torch.no_grad():
    _, X_train_compressed = model(X_train_tensor, return_last_h=True)
    _, X_valid_compressed = model(X_valid_tensor, return_last_h=True)
    _, X_test_compressed = model(X_test_tensor, return_last_h=True)

X_train_compressed = X_train_compressed.cpu().numpy().squeeze()
X_valid_compressed = X_valid_compressed.cpu().numpy().squeeze()
X_test_compressed = X_test_compressed.cpu().numpy().squeeze()

print("Final shape of X_train_compressed:", X_train_compressed.shape)
# Should be (4000, latent_dim)

Final shape of X_train_compressed: (4000, 60)


In [74]:
results_lr = train_and_evaluate_model('lr', X_train_compressed, X_valid_compressed, X_test_compressed, y_train, y_valid, y_test)

In [75]:
results_lr

{'validation': {'auroc': 0.8265470591614956, 'auprc': 0.44414492777434544},
 'test': {'auroc': 0.8237008672147763, 'auprc': 0.4481744592623669}}

## Q3.2 Simulate label scarcity

In [80]:
# Define patient sample sizes
sample_sizes = [100, 500, 1000]

- Train three different supervised (as in Q2.x) models with the same (or as similar as
possible) architecture as your pretrained network, but only use 100, 500, and 1000
patients from the training set and report your full test set performance.

In [81]:
# Store results
results = {}

# Loop through different sample sizes
for sample_size in sample_sizes:
    print(f"\nTraining with sample size: {sample_size}")

    # Subsample training data
    indices = np.random.choice(len(X_train), sample_size, replace=False)
    X_train_sub = X_train[indices]
    y_train_sub = y_train[indices]

    # Build LSTM model
    lstm_model = build_lstm_model((X_train.shape[1], X_train.shape[2]))

    # Train the model
    history = lstm_model.fit(
        X_train, y_train_sub,
        validation_data=(X_valid, y_valid),
        epochs=30,
        batch_size=32,
        verbose=1,
        callbacks=[
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
        ]
    )

    # Evaluate on test set
    y_test_pred = lstm_model.predict(X_test).ravel()  # Flatten predictions
    test_auroc = roc_auc_score(y_test, y_test_pred)
    test_auprc = average_precision_score(y_test, y_test_pred)

    # Store results
    results[sample_size] = {"AUROC": test_auroc, "AUPRC": test_auprc}

    # Print results
    print(f"Performance with {sample_size} samples:")
    print(f"AUROC: {test_auroc:.4f} | AUPRC: {test_auprc:.4f}")

# Print summary of all runs
print("\nSummary of Results:")
for size, metrics in results.items():
    print(f"Sample Size {size}: AUROC = {metrics['AUROC']:.4f}, AUPRC = {metrics['AUPRC']:.4f}")


Training with sample size: 100


AttributeError: 'numpy.ndarray' object has no attribute 'drop'

- Train three linear probes (as in Q3.1 step 2) using only 100, 500, 1000 labelled
patients and report the full test set C performance.

In [293]:
# Store results
results_lr = {}

for num_patients in sample_sizes:
    # Select a random subset of patients from X_train_compressed
    indices = np.random.choice(len(X_train_compressed), num_patients, replace=False)
    X_train_subset = X_train_compressed[indices]
    y_train_subset = y_train[indices]

    # Train and evaluate the model
    results_lr[num_patients] = train_and_evaluate_model('lr', X_train_subset, X_valid_compressed, X_test_compressed, y_train_subset, y_valid, y_test)

# Print results
for num, res in results_lr.items():
    print(f"Train Size: {num}")
    print(f"  Validation AUROC: {res['validation']['auroc']:.4f}, AUPRC: {res['validation']['auprc']:.4f}")
    print(f"  Test AUROC: {res['test']['auroc']:.4f}, AUPRC: {res['test']['auprc']:.4f}")
    print("-" * 50)

Train Size: 100
  Validation AUROC: 0.5923, AUPRC: 0.1948
  Test AUROC: 0.6166, AUPRC: 0.2341
--------------------------------------------------
Train Size: 500
  Validation AUROC: 0.7510, AUPRC: 0.3292
  Test AUROC: 0.7532, AUPRC: 0.3492
--------------------------------------------------
Train Size: 1000
  Validation AUROC: 0.8001, AUPRC: 0.3948
  Test AUROC: 0.7917, AUPRC: 0.3917
--------------------------------------------------


## Q.3.3 Visualising Learned Representations

Visualize your learned representations through a dimensionality reduction technique such as
t-SNE7 or UMAP8 (2 Pt).

Are data points with different labels distributed identically? (1 pt)


Use a quantitative clustering metric to assess the quality of your dimensionality reduction
w.r.t. target class labels (1 pt).

In [285]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

def visualize_embeddings(embeddings, labels, method="tsne"):
    """
    Visualizes learned representations using t-SNE or UMAP.

    Args:
        embeddings (numpy array): High-dimensional patient embeddings (N, D).
        labels (numpy array): Target class labels (N,).
        method (str): "tsne" or "umap" for dimensionality reduction.
    """
    if method == "tsne":
        reducer = TSNE(n_components=2, perplexity=30, random_state=42)
    elif method == "umap":
        reducer = umap.UMAP(n_components=2, random_state=42)
    else:
        raise ValueError("Method must be 'tsne' or 'umap'")

    reduced_embeddings = reducer.fit_transform(embeddings)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], c=labels, cmap='viridis', alpha=0.7)
    plt.colorbar(scatter, label="Class Labels")
    plt.title(f"Visualization of Learned Representations ({method.upper()})")
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.show()

    return reduced_embeddings

def evaluate_clustering(reduced_embeddings, labels):
    """
    Evaluates clustering quality of the reduced embeddings.

    Args:
        reduced_embeddings (numpy array): 2D embeddings from t-SNE or UMAP.
        labels (numpy array): True class labels.

    Returns:
        dict: Clustering metrics (Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz Index).
    """
    silhouette = silhouette_score(reduced_embeddings, labels)
    davies_bouldin = davies_bouldin_score(reduced_embeddings, labels)
    calinski_harabasz = calinski_harabasz_score(reduced_embeddings, labels)

    return {
        "Silhouette Score": silhouette,
        "Davies-Bouldin Index": davies_bouldin,
        "Calinski-Harabasz Index": calinski_harabasz
    }

# Example Usage:
# Assume `patient_embeddings` is your (N, D) matrix and `patient_labels` are the true labels.
reduced_data = visualize_embeddings(patient_embeddings, patient_labels, method="tsne")
clustering_metrics = evaluate_clustering(reduced_data, patient_labels)
print(clustering_metrics)


AttributeError: module 'tensorflow._api.v2.compat.v2.__internal__' has no attribute 'register_load_context_function'

# -------------------

In [283]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define Autoencoder
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(LSTMAutoencoder, self).__init__()

        # Encoder
        self.encoder_lstm = nn.LSTM(input_dim, 64, batch_first=True)
        self.encoder_fc = nn.Linear(64, latent_dim)  # Compress to latent_dim

        # Decoder
        self.decoder_fc = nn.Linear(latent_dim, 64)
        self.decoder_lstm = nn.LSTM(64, input_dim, batch_first=True)

    def forward(self, x):
        _, (h_n, _) = self.encoder_lstm(x)
        latent = self.encoder_fc(h_n[-1])  # Take last hidden state

        # Decode
        # decoded = self.decoder_fc(latent).unsqueeze(1).repeat(1, x.size(1), 1)  # Repeat across time steps
        # decoded, _ = self.decoder_lstm(decoded)

        decoded_input = self.decoder_fc(latent).unsqueeze(1)  # Initial decoder input
        decoded, _ = self.decoder_lstm(decoded_input)  # Let LSTM handle the sequence


        return decoded, latent  # Return both decoded and encoded representation

# Model & Params
input_dim = 40
latent_dim = 40 # Latent representation size
model = LSTMAutoencoder(input_dim, latent_dim).to(device)

# Loss & Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

In [284]:
# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_reshaped, dtype=torch.float32).to(device)
X_valid_tensor = torch.tensor(X_valid_reshaped, dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test_reshaped, dtype=torch.float32).to(device)

# Training loop
num_epochs = 30
batch_size = 32
dataloader = torch.utils.data.DataLoader(X_train_tensor, batch_size=batch_size, shuffle=False)

for epoch in range(num_epochs):
    for batch in dataloader:
        batch = batch.to(device)
        optimizer.zero_grad()

        reconstructed, _ = model(batch)
        loss = criterion(reconstructed, batch)  # Reconstruction loss
        loss.backward()
        optimizer.step()

    # Validation Loss
    with torch.no_grad():
        val_reconstructed, _ = model(X_valid_tensor)
        val_loss = criterion(val_reconstructed, X_valid_tensor)

    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")


/Users/tnorlha/miniconda3/lib/python3.12/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32, 49, 40])) that is different to the input size (torch.Size([32, 1, 40])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/Users/tnorlha/miniconda3/lib/python3.12/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([4000, 49, 40])) that is different to the input size (torch.Size([4000, 1, 40])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [1/30], Train Loss: 0.9201, Val Loss: 2.1894
Epoch [2/30], Train Loss: 0.8869, Val Loss: 2.1579
Epoch [3/30], Train Loss: 0.8671, Val Loss: 2.1324
Epoch [4/30], Train Loss: 0.8387, Val Loss: 2.1148
Epoch [5/30], Train Loss: 0.8224, Val Loss: 2.0942
Epoch [6/30], Train Loss: 0.8039, Val Loss: 2.0851
Epoch [7/30], Train Loss: 0.7934, Val Loss: 2.0732
Epoch [8/30], Train Loss: 0.7821, Val Loss: 2.0643
Epoch [9/30], Train Loss: 0.7716, Val Loss: 2.0590
Epoch [10/30], Train Loss: 0.7641, Val Loss: 2.0568
Epoch [11/30], Train Loss: 0.7563, Val Loss: 2.0525
Epoch [12/30], Train Loss: 0.7616, Val Loss: 2.0513
Epoch [13/30], Train Loss: 0.7559, Val Loss: 2.0471
Epoch [14/30], Train Loss: 0.7502, Val Loss: 2.0458
Epoch [15/30], Train Loss: 0.7508, Val Loss: 2.0439
Epoch [16/30], Train Loss: 0.7509, Val Loss: 2.0428
Epoch [17/30], Train Loss: 0.7504, Val Loss: 2.0430
Epoch [18/30], Train Loss: 0.7485, Val Loss: 2.0403
Epoch [19/30], Train Loss: 0.7483, Val Loss: 2.0410
Epoch [20/30], Train 

In [176]:
with torch.no_grad():
    _, X_train_compressed = model(X_train_tensor)
    _, X_valid_compressed = model(X_valid_tensor)
    _, X_test_compressed = model(X_test_tensor)

X_train_compressed = X_train_compressed.cpu().numpy()
X_valid_compressed = X_valid_compressed.cpu().numpy()
X_test_compressed = X_test_compressed.cpu().numpy()

print("Final shape of X_train_compressed:", X_train_compressed.shape)
# Should be (4000, latent_dim)

Final shape of X_train_compressed: (4000, 40)


In [177]:
results_rf = train_and_evaluate_model('rf', X_train_compressed, X_valid_compressed, X_test_compressed, y_train, y_valid, y_test)

In [178]:
results_lr = train_and_evaluate_model('lr', X_train_compressed, X_valid_compressed, X_test_compressed, y_train, y_valid, y_test)

In [179]:
results_rf

{'validation': {'auroc': 0.7939155914508028, 'auprc': 0.41022750691012766},
 'test': {'auroc': 0.7942325837494213, 'auprc': 0.42642024308333976}}

In [180]:
results_lr

{'validation': {'auroc': 0.8234922354640665, 'auprc': 0.4454574949203807},
 'test': {'auroc': 0.8225686075759283, 'auprc': 0.4640215936702659}}